# Deterministic Rules vs. Fine-Tuned AnyMatch — Full Population

Run on the **AllianceChicago VM** (PHI). Head-to-head of the two scorers over the *same*
candidate pairs:

- **Rules** — `deterministic_rules_predictions_full_v1.csv` (`rule_pred` ∈ match / non_match / review, + `rule_id`, `rule_reason`).
- **Model** — `anymatch_predictions_full_v3.csv` (`pred` ∈ 0/1, `match_prob` ∈ [0,1]).

Still **no ground truth**, so this is about *where and why they disagree*, not who is "right".
Two questions drive it:
1. On pairs the rules confidently decide, does the model agree? Disagreements are leads on
   either a model miss or a rule miss — we read the records to judge.
2. How does the model score the pairs the rules sent to **review**? If the model is confidently
   split there, it's overconfident on genuinely-hard cases; if it hovers near 0.5, it's
   appropriately uncertain on the same pairs the rules flagged.

In [ ]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
# Run from the AnyMatch/ repo root. If launched inside deterministic_rules/, hop up one level.
if not os.path.exists('loo.py') and os.path.basename(os.getcwd()) == 'deterministic_rules':
    os.chdir('..')
assert os.path.exists('loo.py'), f'Run from the AnyMatch/ repo root (cwd={os.getcwd()!r}).'
print('cwd:', os.getcwd())
sys.path.insert(0, 'deterministic_rules')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.alliance_schema import id_str_dtypes, prep_record_df, FEATURE_COLS
import rules_alliance as R

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 160)

RULES_CSV       = 'data/alliance/deterministic_rules_predictions_full_v1.csv'
MODEL_CSV       = 'data/alliance/anymatch_predictions_full_v3.csv'
RECORDS_PARQUET = 'data/alliance/MDM_Population_cleaned_v4_2026_06_16.parquet'

In [ ]:
rules = pd.read_csv(RULES_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
model = pd.read_csv(MODEL_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
print(f'rules: {len(rules):,}   model: {len(model):,}')

df = rules.merge(model[['PATID_A', 'PATID_B', 'pred', 'match_prob']],
                 on=['PATID_A', 'PATID_B'], how='inner')
print(f'joined on PATID pair: {len(df):,}')
miss_r = len(rules) - len(df); miss_m = len(model) - len(df)
if miss_r or miss_m:
    print(f'  note: {miss_r:,} rule rows and {miss_m:,} model rows had no counterpart (dropped).')
df['model_pred'] = df['pred'].map({1: R.MATCH, 0: R.NON_MATCH})
df.head()

In [ ]:
# records for joining fields back into examples
records = pd.read_parquet(RECORDS_PARQUET)
for col, dt in id_str_dtypes(records.columns).items():
    records[col] = records[col].astype(dt)
recs = prep_record_df(records[records['valid_record']].copy()).set_index('PATID')
print(f'{len(recs):,} valid records indexed.')

SHOW_FIELDS = ['first_name', 'middle_name', 'last_name', 'suffix', 'dob', 'ssn', 'ssn4',
               'sex', 'address', 'city', 'state', 'zip', 'phone', 'email']

def show_pair(row):
    a = recs.loc[row.PATID_A, SHOW_FIELDS]; b = recs.loc[row.PATID_B, SHOW_FIELDS]
    cmp = pd.DataFrame({'A': a, 'B': b})
    cmp['same'] = (cmp['A'].astype(str).str.upper().str.strip() == cmp['B'].astype(str).str.upper().str.strip())
    print(f'A={row.PATID_A} B={row.PATID_B} | RULES={row.rule_pred} [{row.rule_id}] | '
          f'MODEL={row.model_pred} (p={row.match_prob:.3f})')
    print(cmp.to_string()); print('-' * 80)

def show_rows(sub, k=6, seed=0):
    if not len(sub): print('(none)'); return
    for r in sub.sample(min(k, len(sub)), random_state=seed).itertuples():
        show_pair(r)

## 1. Agreement matrix (rules 3-way × model 2-way)

The whole comparison in one table. Diagonal-ish cells (rules match / model match,
rules non_match / model non_match) are agreement. The `review` row shows how the model
splits the pairs the rules declined to decide.

In [ ]:
ct = pd.crosstab(df['rule_pred'], df['model_pred']).reindex(
    index=[R.MATCH, R.NON_MATCH, R.REVIEW], columns=[R.MATCH, R.NON_MATCH], fill_value=0)
print('Counts:'); print(ct.to_string())
print('\nRow-normalized (%):'); print((ct.div(ct.sum(1), axis=0) * 100).round(1).to_string())

# Agreement on the rules' DECIDED subset (drop review; map both to binary)
dec = df[df['rule_pred'] != R.REVIEW].copy()
agree = (dec['rule_pred'] == dec['model_pred']).mean()
print(f'\nDecided-subset agreement (rules match/non_match vs model): {agree:.3%} of {len(dec):,} pairs')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
im = ax.imshow(ct.values, cmap='Blues')
ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns)
ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index)
ax.set_xlabel('MODEL'); ax.set_ylabel('RULES'); ax.set_title('Rules vs Model — pair counts')
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, f'{ct.values[i, j]:,}', ha='center', va='center',
                color='white' if ct.values[i, j] > ct.values.max() / 2 else 'black')
plt.tight_layout(); plt.show()

## 2. match_prob distribution by rule decision

Where does the model's probability mass land within each rule bucket? Ideal picture:
rule-`match` pairs pile near 1.0, rule-`non_match` near 0.0, and rule-`review` spread out /
centered — i.e. the model is *also* uncertain exactly where the rules abstain.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for a, dec_name, color in zip(ax, [R.MATCH, R.NON_MATCH, R.REVIEW], ['#2ca02c', '#d62728', '#ff7f0e']):
    sub = df[df['rule_pred'] == dec_name]['match_prob']
    a.hist(sub, bins=40, color=color)
    a.set_title(f'rule = {dec_name}  (n={len(sub):,})')
    a.set_xlabel('model match_prob'); a.axvline(0.5, ls='--', c='k', lw=1)
ax[0].set_ylabel('pairs'); plt.tight_layout(); plt.show()

print('model match_prob summary by rule decision:')
print(df.groupby('rule_pred')['match_prob'].describe()[['mean', '25%', '50%', '75%']].round(3).to_string())

## 3. How does the model treat the rules' REVIEW queue?

These are the pairs the deterministic engine refused to auto-decide. If the model resolves
them confidently (prob near 0 or 1), it's making calls the rules deemed unsafe — worth
spot-checking whether those calls are trustworthy or overconfident (e.g. twins it merges).

In [ ]:
rev = df[df['rule_pred'] == R.REVIEW].copy()
print(f'Review queue: {len(rev):,} pairs')
print(f'  model calls MATCH:     {(rev.pred == 1).sum():,} ({(rev.pred == 1).mean():.1%})')
print(f'  model calls NON-MATCH: {(rev.pred == 0).sum():,} ({(rev.pred == 0).mean():.1%})')
conf = ((rev.match_prob < 0.1) | (rev.match_prob > 0.9)).mean()
print(f'  model is CONFIDENT (p<0.1 or p>0.9) on {conf:.1%} of the review queue')

# Model confidently MATCHES a rules-review pair: the dangerous auto-merges to inspect
print('\n### Rules=REVIEW, model confidently MATCH (p>0.9) — does the model over-merge here? ###')
show_rows(rev[rev.match_prob > 0.9], k=6, seed=11)

In [ ]:
print('### Rules=REVIEW, model confidently NON-MATCH (p<0.1) ###')
show_rows(rev[rev.match_prob < 0.1], k=5, seed=12)

## 4. Hard disagreements on the decided subset

The most informative cases: where one scorer says match and the other says non-match. Each is
a candidate miss for *somebody*. We read the records to judge which call looks right and feed
that back into rule refinement (or model-trust caveats).

In [ ]:
rules_match_model_non = df[(df.rule_pred == R.MATCH) & (df.model_pred == R.NON_MATCH)]
rules_non_model_match = df[(df.rule_pred == R.NON_MATCH) & (df.model_pred == R.MATCH)]
print(f'RULES match / MODEL non-match: {len(rules_match_model_non):,}  '
      f'(possible MODEL miss, or a rule over-merge)')
print(f'RULES non-match / MODEL match: {len(rules_non_model_match):,}  '
      f'(possible RULE miss, or a model false-merge)')

In [ ]:
# Which rule_ids generate the rules-match / model-non-match disagreement?
print('rule_id behind RULES=match, MODEL=non-match:')
print(rules_match_model_non['rule_id'].value_counts().to_string())
print('\nExamples (model says these are NOT the same person — is the rule wrong?):')
show_rows(rules_match_model_non, k=8, seed=21)

In [ ]:
print('rule_id behind RULES=non-match, MODEL=match:')
print(rules_non_model_match['rule_id'].value_counts().to_string())
print('\nExamples (model merges these — did the rules miss a true match?):')
show_rows(rules_non_model_match, k=8, seed=22)

## 5. Strong agreement, for contrast

Both call match (high-confidence merges everyone agrees on) and both call non-match — to
confirm the bulk of the population is uncontroversial and the disagreements above are the tail.

In [ ]:
both_match = df[(df.rule_pred == R.MATCH) & (df.model_pred == R.MATCH)]
both_non   = df[(df.rule_pred == R.NON_MATCH) & (df.model_pred == R.NON_MATCH)]
print(f'both MATCH: {len(both_match):,}   |   both NON-MATCH: {len(both_non):,}')
print('\n### Both MATCH (consensus merges) ###')
show_rows(both_match, k=4, seed=31)

## 6. Takeaways

Fill in after running:
- Decided-subset agreement rate; is it high (the two methods mostly concur)?
- Shape of `match_prob` within each rule bucket — does the model's uncertainty line up with the review queue?
- On the review queue: is the model confident or uncertain, and do its confident merges look safe (esp. twins/same-name-DOB)?
- The two disagreement buckets: which rule_ids drive them, and from the examples, who looks right.
- Concrete follow-ups: rule refinements, or pairs/segments where the model shouldn't be trusted unsupervised.